In [2]:
import json
import re

In [18]:
datapath = r'data\protokolle\12_053_1991-11-06.json'
datapath = r'data\protokolle\20_012_2022-01-14.json'

with open(datapath) as f:
    d = json.load(f)
    text = d['text']

text = re.sub(r'BÜNDNIS(?:SES)?\s*90\/DIE\s*GRÜNEN', 'GRÜNE', text) # That's a long short form for a party...
text = re.sub(r'\n-\n', '-', text) # weird formatting probably due to digitalization from printed mediums 

In [43]:
name_match = r'((?:Dr\.\s)?(?:\w+(?:-\w+)?\s)+\w+(?:-\w+)?)' # Dr. Marie-Agnes Strack-Zimmermann...
name_match = r'((?:Dr\.\s)?(?:\w+(?:-\w+)?\s){1,3}\w+(?:-\w+)?)'
parties = ['CDU/CSU', 'GRÜNE','SPD', 'FDP', 'AfD', 'DIE LINKE', 'KPD', 'BP', 'DP', 'WAV', 'Z', 'fraktionslos' ] # Die Abkürzungen der wichtigsten - auch historischen - Parteien
partei_match = r'(CDU\/CSU|CSU|CDU|GRÜNE|FDP|AfD|SPD|KPD|BP|DP|WAV|Z|DIE LINKE|fraktionslos)'
kommentar_match = rf'{name_match}\s+\[{partei_match}\]:\s([\s\S]*?)[-–—\)]'
beifall_match = r'[(?:-–—\s)\(]Beifall [\s\S]*?[-–—\)]'
zuruf_match = rf'[(?:-–—\s)\(]Zuruf .*?{partei_match}: ([\s\S]*?)[-–—\)]'
speaker_match = rf'{name_match}\s\({partei_match}\):|\n{name_match}, ([\w\s]*?)(?:Saatssekretär|Staatsminister|Bundesminister)[\w\s]*?\n?[\w\s]*?:' # right side has two capture-groups too so that the alignment of names is not out of order when splitting


In [44]:
testtext = '''Johannes Wild g .
Lassen Sie mich kurz skizzieren, was ich als Bundesministerin für wirtschaftliche Zusammenarbeit und Entwicklung vorhabe: Im Koalitionsvertrag steht, dass '''
re.findall(name_match, testtext, flags= re.UNICODE)

['Johannes Wild g',
 'Lassen Sie mich kurz',
 'was ich als Bundesministerin',
 'für wirtschaftliche Zusammenarbeit und',
 'Entwicklung vorhabe',
 'Im Koalitionsvertrag steht']

In [28]:
print(speaker_match)

((?:Dr\.\s)?(?:\w+(?:-\w+)?\s)+\w+(?:-\w+)?|\w+(?:\s\w+)*)\s\((CDU\/CSU|CSU|CDU|GRÜNE|FDP|AfD|SPD|KPD|BP|DP|WAV|Z|DIE LINKE|fraktionslos)\):|\n((?:Dr\.\s)?(?:\w+(?:-\w+)?\s)+\w+(?:-\w+)?|\w+(?:\s\w+)*), ([\w\s]*?)(?:Saatssekretär|Staatsminister|Bundesminister)[\w\s]*?\n?[\w\s]*?:


### Get List of all Representatives and their party affiliation since 1949
see Notebook get_abgeordneten_data for more

In [11]:
import csv
bt_tuples = []
with open('./abgeordnete.csv', 'r', newline= '',encoding='utf8') as file:
    reader = csv.reader(file)
    next(reader) # skip first row
    for row in reader:
        bt_tuples.append(tuple(row))
bt_tuples[:5]

[('Dieter Janecek', 'GRÜNE', '20'),
 ('Carsten Brodesser', 'CDU', '20'),
 ('Kirsten Kappert-Gonther', 'GRÜNE', '20'),
 ('Bettina Margarethe Wiesmann', 'CDU', '20'),
 ('Sanae Abdi', 'SPD', '20')]

In [26]:
def find_party_by_name(name, tuple_list):
    tuple_list_sorted = sorted(tuple_list, key = lambda x: x[2], reverse = True) # we focus on 
    if name.startswith('Dr.'):
        name = name[4:]
    for item in tuple_list_sorted:
        if name in item[0]:
            if item[1] != 'CSU' and item[1] != 'CDU':
                return item[1]
            else:
                return 'CDU/CSU'  
    return "Party not found"

print(find_party_by_name('Dr. Robert Habeck', bt_tuples))

GRÜNE


In [45]:
count_small = 0 #debugging

speeches_raw = re.split(speaker_match, text, flags=re.UNICODE)[1:]
speeches = []
for i in range(0, len(speeches_raw), 5):
    if speeches_raw[i] != None and speeches_raw[i+1] != None:
        name = speeches_raw[i].strip()    
        party = speeches_raw[i+1].strip()
    else:
        name = speeches_raw[i+2].strip()
        party = find_party_by_name(name, bt_tuples)
        if party == "Party not found":
            print(f'Party not found for {name}')

    speech = {
        'speaker':{
            'name':name,
            'party':party
        },
        'text': re.split(r'(\nVizepräs.{0,99}?:)|(\nPräsid.{0,99}?:)|(\nAnlage)',speeches_raw[i+4])[0].strip() # handle end of File and interruptions by the Bundestagspräsident(in) or Vice Bundestagspräsident(in)
    }


    #debugging
    if party not in parties and party != 'Party not found':
        print(f"möglicher Fehler bei {speech}")

    if len(speech['text']) < 150:
        count_small += 1
    #end_debugging
    
    if len (speeches) >=1 and speeches[-1]['speaker'] == speech['speaker']:
        speeches[-1]['text'] += '\n' + speech['text'] # append interrupted speeches by same speaker e.g. after interruptions by Bundestagspräsident
    else:
        speeches.append(speech)

print(len(speeches))

74


In [46]:
for speech in speeches:
    comments = []
    # comments with known speaker
    for match in re.finditer(kommentar_match,speech['text']):
        comment = {
            'commentator': {
                'name': match.group(1),
                'party': match.group(2)
            },
            'text': match.group(3).strip(),
            'preceding_context': speech['text'][:match.start()] # include all text until comment for later training of LLM
        }
        comments.append(comment)
    
    # comments with unknown speaker
    for match in re.finditer(zuruf_match, re.sub('der LINKEN', 'DIE LINKE', speech['text'])):
        comment = {'commentator':{
                'name': '<unknown>',
                'party': match.group(1)
            },
            'text': match.group(2),
            'preceding_context': speech['text'][:match.start()]
        }
        comments.append(comment)

    speech['comments'] = comments

    # applause
    beifall = re.findall(beifall_match, speech['text'])
    beifall = ''.join(beifall)
    beifall = re.sub('der LINKEN', 'DIE LINKE', beifall)
    beifall_counts = {party: beifall.count(f' {party}') for party in parties}
    speech['applause'] = beifall_counts

In [16]:
speeches[:5]

[{'speaker': {'name': 'Tessa Ganserer', 'party': 'GRÜNE'},
  'text': 'Sehr geehrte Frau Präsidentin! Liebe Kolleginnen und Kollegen! Wenn wir uns die Halbzeitbilanz der Agenda 2030 der Vereinten Nationen anschauen, dann fällt diese, gelinde gesagt, äußerst ernüchternd aus. In den vergangenen Jahren ist die nachhaltige Entwicklung coronabedingt und auch durch den fürchterlichen Angriffskrieg von Russland gegenüber der Ukraine global ins Stocken geraten. Nur 15\xa0Prozent der Indikatoren weisen eine positive Richtung aus. Bei rund der Hälfte der Indikatoren ist die Zielerreichung unwahrscheinlich, auch wenn es leichte Fortschritte gibt. Bei 30\xa0Prozent der einzelnen Bereiche ist Stillstand oder sogar eine rückläufige Entwicklung zu verzeichnen.\nDie Herausforderungen nehmen weiter zu. Im Hinblick auf die Erhaltung der natürlichen Lebensgrundlagen haben wir sechs von neun planetaren Grenzen bereits überschritten. Aber auch im Bereich soziale Gerechtigkeit ist die Entwicklung äußerst neg

In [47]:
json_string = json.dumps(speeches, indent=4) 

# Write JSON string to a file
with open("./data/test/parsed_2022_example.json", "w") as json_file:
    json_file.write(json_string)

# For all Plenarprotokolle in a given folder

In [11]:
import os

In [12]:
data_path = './data/protokolle/'

In [13]:
for protokoll in os.listdir(data_path):
    print(f"filepath: {data_path+protokoll}")
    with open(data_path+protokoll) as f:
        doc = json.load(f)
        text = doc['text']

    speeches_raw = re.split(speaker_match, text)[1:]
    speeches = []
    for i in range(0, len(speeches_raw), 5):
        if speeches_raw[i] != None and speeches_raw[i+1] != None:
            name = speeches_raw[i].strip()    
            party = speeches_raw[i+1].strip()
        else:
            name = speeches_raw[i+2].strip()
            party = find_party_by_name(name, bt_tuples)
            if party == "Party not found":
                print(f'Party not found for {name}')

        speech = {
            'speaker':{
                'name':name,
                'party':party
            },
            'text': re.split(r'(\nVizepräs.{0,99}?:)|(\nPräsid.{0,99}?:)|(:\n)',speeches_raw[i+4])[0].strip() # sometimes the speakers get interrupted by the Bundestagspräsident or Vice Bundestagspräsident
            #'text': re.split(r'(:\n)|(\nAnlage)',speeches_raw[i+4])[0].strip()
        }

        ################
        # comments
        ################

        comments = []
        # comments with known speaker
        for match in re.finditer(kommentar_match,speech['text']):
            comment = {
                'commentator': {
                    'name': match.group(1),
                    'party': match.group(2)
                },
                'text': match.group(3),
                'preceding_context': speech['text'][:match.start()] 
            }
            comments.append(comment)
        
        # comments with unknown speaker
        for match in re.finditer(zuruf_match, re.sub('der LINKEN', 'DIE LINKE', speech['text'])):
            comment = {'commentator':{
                    'name': '<unknown>',
                    'party': match.group(1)
                },
                'text': match.group(2),
                'preceding_context': speech['text'][:match.start()]
            }
            comments.append(comment)

        speech['comments'] = comments

        ################
        # applause
        ################

        beifall = re.findall(beifall_match, speech['text'])
        beifall = ''.join(beifall)
        beifall = re.sub('der LINKEN', 'DIE LINKE', beifall)
        beifall_counts = {party: beifall.count(f' {party}') for party in parties}
        speech['applause'] = beifall_counts

        speeches.append(speech)

    json_string = json.dumps(speeches, indent=4) 
    path = f'./data/parsed_comments/{doc['wahlperiode']}_{doc['dokumentnummer'].split(r'/')[1].zfill(3)}_{doc['datum']}_parsed.json'
    # Write JSON string to a file
    with open(path, "w") as json_file:
        json_file.write(json_string)   


filepath: ./data/protokolle/16_071_2006-12-01.json
filepath: ./data/protokolle/1_172_1951-11-07.json
filepath: ./data/protokolle/1_173_1951-11-08.json
filepath: ./data/protokolle/20_010_2022-01-12.json
Party not found for Nancy Faeser
Party not found for Nancy Faeser
filepath: ./data/protokolle/20_011_2022-01-13.json
Party not found for Anne Spiegel
Party not found for Klara Geywitz
filepath: ./data/protokolle/20_012_2022-01-14.json
filepath: ./data/protokolle/20_148_2024-01-19.json


In [15]:
kommentare = re.findall(kommentar_match, text)
kommentare

[('Andreas Bleck',
  'AfD',
  'Ihr habt Deutschland abgewirtschaftet! Ihr werdet bei der nächsten Bundestagswahl abgestraft!\xa0'),
 ('Albrecht Glaser', 'AfD', 'Dreckiger Schmutz!'),
 ('Andreas Bleck', 'AfD', 'Das macht ihr! Noch nie ging es uns so schlecht!'),
 ('Andreas Bleck', 'AfD', 'Das entscheiden immer noch die Wähler!'),
 ('Andreas Bleck', 'AfD', 'Vor allem!'),
 ('Alexander Dobrindt', 'CDU/CSU', 'So ist es!'),
 ('Jakob Blankenburg',
  'SPD',
  'Wer war denn die letzten Jahre Landwirtschaftsminister?'),
 ('Bettina Hagedorn', 'SPD', 'Bei Ihnen passt gar nichts zusammen!'),
 ('Dr.\xa0Rainer Kraft', 'AfD', 'Von „Correctiv“!'),
 ('Dr.\xa0Rainer Kraft', 'AfD', 'Unbewiesene Behauptung!'),
 ('Dr.\xa0Rainer Kraft',
  'AfD',
  'Das haben sie vor 20\xa0Jahren auch schon gesagt!'),
 ('Dr.\xa0Rainer Kraft',
  'AfD',
  'Wozu brauchen die Eisbrecher, wenn das Eis weg ist?'),
 ('Johannes Schraps', 'SPD', 'Sehr richtig!'),
 ('Dr.\xa0Karamba Diaby', 'SPD', 'Halb voll!'),
 ('Dr.\xa0Karamba Diaby'

In [16]:
import json

In [44]:
with open(r'data\protokolle\9_063_1981-11-11.json') as f:
    d = json.load(f)
    text = d['text']
    text = re.sub(r'BÜNDNIS(?:SES)?\s*90\/DIE\s*GRÜNEN', 'GRÜNE', text) # That's a long short form for a party...
    text = re.sub(r'\n-\n', '-', text) # weird formatting probably due to digitalization from printed mediums   

In [22]:
with open("./data/test/2022_test.txt" , 'w', encoding='utf8') as f:
    f.write(text)

In [19]:
satzende_match = r'(?<!\b(?:Dr|med|z\.B|etc)).\s+(?=[A-Z])'

<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:1: SyntaxWarning: invalid escape sequence '\.'
C:\Users\Robin\AppData\Local\Temp\ipykernel_11668\3505235198.py:1: SyntaxWarning: invalid escape sequence '\.'
  satzende_match = '(?<!\b(?:Dr|med|z\.B|etc)).\s+(?=[A-Z])'


# Sonderfälle

manche Redner wie z.B. amtierende Minister, Staatssekretäre etc. fallen aus dem Raster heraus und es steht keine Partei dahinter -> Wir wollen trotzdem die Parteien anmerken  
-> Abgleich mit XML-Stammdatenliste. Hier besonders angenehm: Schema: <p>"&lt;Vorname> &lt;Name>, &lt;Titel>:" </p> -> Das Ganze auch OHNE Doktortitel bei z.B. Dr. Robert Habeck -> Auslesen von Stammdaten xml

Sonderfall Einwürfe bei 1951-Protokollen. und 1971 "(Abg. <nachname>: <text>)
1951: Auch neuer Textanfang: (Dr.)? <nachname> (<partei>) (<Wahlkreis>)? :
1971 + 1981: (Dr.)? <nachname> (<partei>) (<Wahlkreis>)?: